In [ ]:
import os
print(os.listdir("/kaggle/input/datasets/atharv0919/depmap-raw"))

In [ ]:
import os
RAW = "/kaggle/input/datasets/atharv0919/depmap-raw"
print(os.listdir(RAW))

In [ ]:
import os, subprocess

REPO = "/kaggle/working/bioprior"
if not os.path.exists(REPO):
    r = subprocess.run(["git", "clone", "https://github.com/raveena2309/bioprior.git", REPO],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)

P = REPO + "/data/processed/"
print("repo exists:", os.path.exists(REPO))
print("training_table exists:", os.path.exists(P + "training_table.csv"))
print("DepMap upload visible:", os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "none")

In [ ]:
import os, re, numpy as np, pandas as pd

RAW  = "/kaggle/input/datasets/atharv0919/depmap-raw"
P    = "/kaggle/working/bioprior/data/processed/"
EXPR = os.path.join(RAW, "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv")
MUT  = os.path.join(RAW, "OmicsSomaticMutationsMatrixDamaging.csv")
CNV  = os.path.join(RAW, "OmicsCNGeneWGS.csv")
OUT  = "/kaggle/working/cell_features_gene_universe.npz"
for p in (EXPR, MUT, CNV):
    print(os.path.exists(p), p)

def entrez_cols(columns):                       # "EGFR (1956)" -> {1956: "EGFR (1956)"}
    d = {}
    for c in columns:
        m = re.search(r"\((\d+)\)\s*$", str(c))
        if m: d[int(m.group(1))] = c
    return d

def load(path, name):
    df = pd.read_csv(path, low_memory=False)
    print(f"[{name}] shape {df.shape} | first cols {list(df.columns[:6])}")
    if "IsDefaultEntryForModel" in df.columns:
        df = df[df["IsDefaultEntryForModel"] == "Yes"]
    df = df.set_index("ModelID" if "ModelID" in df.columns else df.columns[0])
    df = df[~df.index.duplicated(keep="first")]
    cols = entrez_cols(df.columns)
    print(f"[{name}] cell lines {len(df)} | gene columns {len(cols)}")
    return df, cols

tt = pd.read_csv(P + "training_table.csv")
expr, ec = load(EXPR, "expression")
mut,  mc = load(MUT,  "mutation")
cnv,  cc = load(CNV,  "cnv")

ppi = pd.read_csv(P + "graph_gene_gene_edges.csv")
ppi_genes   = set(ppi.gene_a) | set(ppi.gene_b)
react_genes = set(pd.read_csv(P + "graph_gene_pathway_edges.csv").gene_id)

u = ppi_genes & react_genes;  print("PPI", len(ppi_genes), "| Reactome", len(react_genes), "| both", len(u))
u &= set(ec); print("... and in expression:", len(u))
u &= set(mc); print("... and in mutation:  ", len(u))
u &= set(cc); print("... and in CNV:       ", len(u))
genes = sorted(u)

all_cells = set(tt.cell_line_id)
cells = sorted(all_cells & set(expr.index) & set(mut.index) & set(cnv.index))
print(f"cell lines: training {len(all_cells)} | with expr {len(all_cells & set(expr.index))} | "
      f"with mut {len(all_cells & set(mut.index))} | with cnv {len(all_cells & set(cnv.index))} | with ALL: {len(cells)}")
print("training rows kept:", tt.cell_line_id.isin(cells).sum(), "of", len(tt))

E = expr.loc[cells, [ec[g] for g in genes]].astype("float32").values
E = (E - E.mean(0)) / (E.std(0) + 1e-6)
Mraw = mut.loc[cells, [mc[g] for g in genes]].apply(pd.to_numeric, errors="coerce").fillna(0).values
print("mutation matrix unique values:", np.unique(Mraw)[:6])
M = (Mraw > 0).astype("int8")
Craw = cnv.loc[cells, [cc[g] for g in genes]].astype("float32").values
print("CNV raw value range: min %.2f | median %.2f | max %.2f" % (np.nanmin(Craw), np.nanmedian(Craw), np.nanmax(Craw)))
rel = 2 ** Craw - 1
C = np.zeros(Craw.shape, dtype="int8"); C[rel < 0.5] = -1; C[rel > 2] = 1
print(f"CNV: deleted {np.mean(C==-1):.1%} | neutral {np.mean(C==0):.1%} | amplified {np.mean(C==1):.1%} | NaN {np.mean(np.isnan(Craw)):.1%}")
print(f"mutated entries: {M.mean():.2%}")

np.savez_compressed(OUT, cell_ids=np.array(cells), gene_ids=np.array(genes), expr=E, mut=M, cnv=C)
print("saved", OUT, "| expr shape", E.shape)

In [ ]:
print("raw CNV percentiles (1,5,25,50,75,95,99):",
      np.nanpercentile(Craw, [1, 5, 25, 50, 75, 95, 99]).round(2))

LO, HI = 0.5, 1.5            # <0.5 = lost ~1 of 2 copies; >1.5 = gained ~1 extra copy
C = np.zeros(Craw.shape, dtype="int8")
C[Craw < LO] = -1
C[Craw > HI] = 1             # NaN compares False, so NaN stays neutral (0)
print(f"CNV: deleted {np.mean(C==-1):.1%} | neutral {np.mean(C==0):.1%} | amplified {np.mean(C==1):.1%}")

np.savez_compressed(OUT, cell_ids=np.array(cells), gene_ids=np.array(genes),
                    expr=E, mut=M, cnv=C)
print("re-saved", OUT, "| shapes:", E.shape, M.shape, C.shape)

In [ ]:
import numpy as np, pandas as pd

P  = "/kaggle/working/bioprior/data/processed/"
tt = pd.read_csv(P + "training_table.csv")
dg = pd.read_csv(P + "graph_drug_gene_edges.csv")
feat  = np.load("/kaggle/working/cell_features_gene_universe.npz")
cells = set(feat["cell_ids"].tolist())

print("1) original training table:", len(tt), "rows |", tt.drug_id.nunique(), "drugs |", tt.cell_line_id.nunique(), "cell lines")

# a) keep only cell lines that have all 3 feature types
t1 = tt[tt.cell_line_id.isin(cells)].copy()
print("2) after keeping the 595 cell lines with full features:", len(t1), "rows |",
      t1.drug_id.nunique(), "drugs |", t1.cell_line_id.nunique(), "cell lines")

# b) how much do duplicated measurements disagree?
dup = t1.duplicated(["drug_id", "cell_line_id"], keep=False)
print("3) rows involved in duplicate (drug, cell line) pairs:", int(dup.sum()))
if dup.any():
    g = t1[dup].groupby(["drug_id", "cell_line_id"])["label"]
    spread = g.max() - g.min()
    print("   LN_IC50 disagreement within duplicates: median %.2f | 90th pct %.2f | max %.2f"
          % (spread.median(), spread.quantile(.9), spread.max()))

# c) average duplicates
df = (t1.groupby(["drug_id", "cell_line_id"], as_index=False)
        .agg(label=("label", "mean"), DRUG_NAME=("DRUG_NAME", "first"), chembl_id=("chembl_id", "first"),
             smiles=("smiles", "first"), PUTATIVE_TARGET=("PUTATIVE_TARGET", "first"),
             PATHWAY_NAME=("PATHWAY_NAME", "first")))

assert not df.duplicated(["drug_id", "cell_line_id"]).any()
assert np.isfinite(df["label"]).all() and df["smiles"].notna().all()

# d) target coverage
n_t = dg.groupby("chembl_id").gene_symbol.nunique()
df["n_targets"]  = df.chembl_id.map(n_t).fillna(0).astype(int)
df["has_target"] = df.n_targets > 0

print("4) FINAL:", len(df), "pairs |", df.drug_id.nunique(), "drugs |", df.cell_line_id.nunique(), "cell lines")
print("   drugs with >=1 target:", df[df.has_target].drug_id.nunique(), "| pairs of those drugs:", int(df.has_target.sum()))
print("   matrix fill: %.1f%% of all drug x cell-line pairs measured"
      % (100 * len(df) / (df.drug_id.nunique() * df.cell_line_id.nunique())))
print("   LN_IC50 quantiles (0,1,25,50,75,99,100%):", df.label.quantile([0, .01, .25, .5, .75, .99, 1]).round(2).tolist())
print("   cell lines per drug (min / median):", df.groupby("drug_id").size().min(), df.groupby("drug_id").size().median())

df.to_csv("/kaggle/working/clean_table.csv", index=False)
print("saved /kaggle/working/clean_table.csv")

In [ ]:
!pip install -q rdkit
import os, glob, subprocess, json, time, re, copy
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

REPO = "/kaggle/working/bioprior"
if not os.path.exists(REPO):
    r = subprocess.run(["git","clone","https://github.com/raveena2309/bioprior.git",REPO],
                       capture_output=True, text=True); print(r.stdout, r.stderr)
P = REPO + "/data/processed/"

def find(name):
    """Search all attached Kaggle datasets and /kaggle/working for a file."""
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"/kaggle/working/{name}")
    assert hits, f"{name} not found. Attach the dataset via Add Input."
    return hits[0]

FEAT = find("cell_features_gene_universe.npz")
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("repo:", os.path.exists(P + "training_table.csv"), "| features:", FEAT, "| device:", dev)

In [ ]:
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import AllChem
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
RDLogger.DisableLog("rdApp.*")

df = pd.read_csv(find("clean_table.csv"))
tt = pd.read_csv(P + "training_table.csv")
cells = set(np.load(FEAT)["cell_ids"].tolist())

# duplicate disagreement per pair (noise flag; 0 for pairs measured once)
t1 = tt[tt.cell_line_id.isin(cells)]
g = t1.groupby(["drug_id","cell_line_id"])["label"]
sp = (g.max() - g.min()).rename("spread").reset_index()
n0 = len(df); df = df.merge(sp, on=["drug_id","cell_line_id"], how="left"); assert len(df) == n0
df["spread"] = df.spread.fillna(0)
print("pairs with spread>2:", int((df.spread > 2).sum()), "of", len(df))

drugs = df.drop_duplicates("drug_id")[["drug_id","DRUG_NAME","smiles"]].reset_index(drop=True)
mols = [Chem.MolFromSmiles(s) for s in drugs.smiles]
assert all(m is not None for m in mols), "unparseable SMILES"
fps = [AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=1024) for m in mols]

THR, rr, cc = 0.5, [], []
for i in range(len(fps)-1):
    for k, s in enumerate(DataStructs.BulkTanimotoSimilarity(fps[i], fps[i+1:])):
        if s >= THR: rr.append(i); cc.append(i+1+k)
n = len(drugs)
n_fam, fam = connected_components(coo_matrix((np.ones(len(rr)), (rr, cc)), shape=(n, n)), directed=False)
drugs["family"] = fam
print(f"drugs {n} -> families {n_fam} | similar pairs >= {THR}: {len(rr)} | largest family: {np.bincount(fam).max()}")
df = df.merge(drugs[["drug_id","family"]], on="drug_id", how="left").reset_index(drop=True)
assert df.family.notna().all()

df["fold"] = -1
for f, (_, te) in enumerate(GroupKFold(n_splits=5).split(df, groups=df.family)):
    df.loc[te, "fold"] = f
assert (df.fold >= 0).all()

for f in range(5):                      # inner validation (for early stopping), also by family
    role = np.where(df.fold == f, "test", "train").astype(object)
    tr = np.where(df.fold != f)[0]
    _, va = next(GroupShuffleSplit(1, test_size=0.1, random_state=f).split(tr, groups=df.family.values[tr]))
    role[tr[va]] = "val"
    df[f"role_{f}"] = role

# ---- leakage tests ----
assert not df.duplicated(["drug_id","cell_line_id"]).any()
assert df.groupby("drug_id").fold.nunique().max() == 1
for f in range(5):
    r = df[f"role_{f}"]
    D = {k: set(df.loc[r==k,"drug_id"]) for k in ["train","val","test"]}
    Fm = {k: set(df.loc[r==k,"family"]) for k in ["train","val","test"]}
    assert not (D["train"]&D["test"]) and not (D["val"]&D["test"]) and not (D["train"]&D["val"])
    assert not (Fm["train"]&Fm["test"]) and not (Fm["val"]&Fm["test"]) and not (Fm["train"]&Fm["val"])
    te = df[r=="test"]
    print(f"fold {f}: drugs {len(D['train'])}/{len(D['val'])}/{len(D['test'])} | pairs {(r=='train').sum()}/{(r=='val').sum()}/{(r=='test').sum()}"
          f" | test drugs with target: {te[te.has_target].drug_id.nunique()}")
print("ALL LEAKAGE CHECKS PASSED")

# ---- baselines: what does "good" mean? ----
def per_drug_r(d, pred, min_n=30):
    d = d.assign(pred=np.asarray(pred))
    return float(np.mean([np.corrcoef(g.pred, g.label)[0,1] for _, g in d.groupby("drug_id")
                          if len(g) >= min_n and g.pred.std() > 0]))
rng = np.random.default_rng(0)
for f in range(5):
    tr, te = df[df[f"role_{f}"]=="train"], df[df[f"role_{f}"]=="test"]
    pred = te.cell_line_id.map(tr.groupby("cell_line_id").label.mean()).fillna(tr.label.mean())
    print(f"fold {f} | cell-line-mean baseline: pooled r={np.corrcoef(pred, te.label)[0,1]:.3f}, per-drug r={per_drug_r(te, pred):.3f}"
          f" | shuffled-label pooled r={np.corrcoef(rng.permutation(te.label.values), te.label)[0,1]:.3f} (~0 expected)")
df.to_csv("/kaggle/working/clean_table_folds.csv", index=False)
print("saved clean_table_folds.csv  <-- DOWNLOAD THIS NOW")

# **cell 2 (stage 3)**

In [ ]:
SEL_G = 1000          # genes kept in the graph. Lower this if you get out-of-memory errors later.
USE_GDSC_TARGETS = False   # False = ChEMBL targets only (matches the proposal)

df = pd.read_csv(find("clean_table_folds.csv"))
feat = np.load(FEAT)
cell_ids = feat["cell_ids"].tolist(); genes_all = feat["gene_ids"].astype(int)
assert (np.diff(genes_all) > 0).all()

# symbol -> Entrez, read from the header only (cheap)
hdr = pd.read_csv(find("OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv"), nrows=0).columns
sym2ent = {m.group(1): int(m.group(2)) for c in hdr if (m := re.match(r"^(.+?) \((\d+)\)$", str(c)))}

drug_ids = sorted(df.drug_id.unique()); didx = {d:i for i,d in enumerate(drug_ids)}
D = len(drug_ids)

# --- drug -> gene edges
dgc = pd.read_csv(P + "graph_drug_gene_edges.csv")
dgc["gene_id"] = dgc.gene_symbol.map(sym2ent)
print(f"ChEMBL target symbols: {dgc.gene_symbol.nunique()} | mapped to Entrez: {dgc.dropna(subset=['gene_id']).gene_symbol.nunique()}")
dgc = dgc.dropna(subset=["gene_id"]); dgc["gene_id"] = dgc.gene_id.astype(int)
dmeta = df.drop_duplicates("drug_id")[["drug_id","chembl_id","PUTATIVE_TARGET"]]
dge = dmeta.merge(dgc[["chembl_id","gene_id"]], on="chembl_id")[["drug_id","gene_id"]]
if USE_GDSC_TARGETS:
    have = set(dge.drug_id); extra = []
    for _, r in dmeta[~dmeta.drug_id.isin(have)].iterrows():
        for s in re.split(r"[,;]", str(r.PUTATIVE_TARGET)):
            if s.strip() in sym2ent: extra.append((r.drug_id, sym2ent[s.strip()]))
    dge = pd.concat([dge, pd.DataFrame(extra, columns=["drug_id","gene_id"])]).drop_duplicates()
univ = set(genes_all)
dge_u = dge[dge.gene_id.isin(univ)]
print(f"drug-gene edges: {len(dge)} -> inside gene universe: {len(dge_u)} | drugs with targets: {dge_u.drug_id.nunique()}/{D}")

# --- PPI (undirected, unique pairs) inside universe, then select genes
ppi = pd.read_csv(P + "graph_gene_gene_edges.csv")
n_ppi_raw = len(ppi)
ppi = ppi[ppi.gene_a.isin(univ) & ppi.gene_b.isin(univ) & (ppi.gene_a != ppi.gene_b)]
und = np.unique(np.sort(ppi[["gene_a","gene_b"]].values, axis=1), axis=0)
deg = pd.concat([pd.Series(und[:,0]), pd.Series(und[:,1])]).value_counts()
tgt = sorted(set(dge_u.gene_id))
fill = [g for g in deg.index if g not in set(tgt)][:max(0, SEL_G - len(tgt))]
genes = sorted(set(tgt) | set(fill)); G = len(genes); gidx = {g:i for i,g in enumerate(genes)}
print(f"genes: all-sources universe {len(univ)} -> selected G={G} (targets {len(tgt)} + top-degree {len(fill)})")

gg = np.array([[gidx[a], gidx[b]] for a, b in und if a in gidx and b in gidx]).T   # unique undirected, [2,E]
dgE = np.array([[didx[d], gidx[g]] for d, g in dge_u[dge_u.gene_id.isin(gidx)].values]).T

# --- pathways (>=5 selected genes) and hierarchy
gp_df = pd.read_csv(P + "graph_gene_pathway_edges.csv")
gp_df = gp_df[gp_df.gene_id.isin(gidx)]
cnt = gp_df.groupby("reactome_id").gene_id.nunique()
keep = cnt[cnt >= 5].index; pidx = {p:i for i,p in enumerate(keep)}; NP = len(pidx)
gp_df = gp_df[gp_df.reactome_id.isin(pidx)]
gpE = np.array([[gidx[g], pidx[p]] for g, p in gp_df[["gene_id","reactome_id"]].values]).T
ph = pd.read_csv(P + "graph_pathway_hierarchy_edges.csv")
ph = ph[ph.parent_pathway.isin(pidx) & ph.child_pathway.isin(pidx)]
ppE = np.array([[pidx[a], pidx[b]] for a, b in ph.values]).T

# --- drug-drug (chemistry, not shuffled)
dd = pd.read_csv(P + "graph_drug_drug_edges.csv"); dd = dd[dd.drug_a.isin(didx) & dd.drug_b.isin(didx)]
ddE = np.array([[didx[a], didx[b]] for a, b in dd[["drug_a","drug_b"]].values]).T if len(dd) else np.zeros((2,0),int)

base = {"dg": dgE, "gg": gg, "gp": gpE, "pp": ppE, "dd": ddE}
SIZES = {"dg":(D,G), "gg":(G,G), "gp":(G,NP), "pp":(NP,NP), "dd":(D,D)}
print("\n=== graph size table (before -> after) ===")
print(f"PPI edges        {n_ppi_raw:>8,} -> {gg.shape[1]:,}")
print(f"gene-pathway     {49642:>8,} -> {gpE.shape[1]:,}  (pathways kept: {NP})")
print(f"pathway hierarchy{2899:>8,} -> {ppE.shape[1]:,}")
print(f"drug-gene        {347:>8,} -> {dgE.shape[1]:,}")
print(f"drug-drug        {5:>8,} -> {ddE.shape[1]:,}")
assert all(e.shape[0] == 2 and e.shape[1] > 0 for k, e in base.items() if k != "dd")

# --- node features and training pairs on GPU
pos = np.searchsorted(genes_all, genes)
CELLX = torch.tensor(np.stack([feat["expr"][:,pos], feat["mut"][:,pos], feat["cnv"][:,pos]], -1).astype("float32"), device=dev)
fp_np = np.stack([np.array(AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(
        df[df.drug_id == d].smiles.iloc[0]), 2, nBits=1024), dtype="float32") for d in drug_ids])
FPS = torch.tensor(fp_np, device=dev)
cmap = {c:i for i,c in enumerate(cell_ids)}; assert df.cell_line_id.isin(cmap).all()
PC = torch.tensor(df.cell_line_id.map(cmap).values, device=dev)
PD = torch.tensor(df.drug_id.map(didx).values, device=dev)
Y = df.label.values.astype("float32")
print("CELLX", tuple(CELLX.shape), "| FPS", tuple(FPS.shape), "| pairs", len(df))

# the model ( cell 3)

In [ ]:
class RelAttn(nn.Module):                      # level 1: node attention for ONE relation
    def __init__(s, hid, heads):
        super().__init__(); s.h, s.f = heads, hid // heads
        s.Ws = nn.Linear(hid, hid, bias=False); s.Wd = nn.Linear(hid, hid, bias=False)
        s.a_s = nn.Parameter(torch.randn(1,1,heads,s.f) * 0.1); s.a_d = nn.Parameter(torch.randn(1,1,heads,s.f) * 0.1)
    def forward(s, hs, hd, ei):                # hs [B,Ns,hid], hd [B,Nd,hid], ei [2,E]
        B, Nd = hs.size(0), hd.size(1); src, dst = ei
        zs = s.Ws(hs).view(B,-1,s.h,s.f); zd = s.Wd(hd).view(B,-1,s.h,s.f)
        e = F.leaky_relu((zs*s.a_s).sum(-1)[:,src] + (zd*s.a_d).sum(-1)[:,dst], 0.2)   # [B,E,H]
        idx = dst.view(1,-1,1).expand_as(e)
        mx = torch.full((B,Nd,s.h), -1e9, device=e.device).scatter_reduce(1, idx, e.detach(), reduce="amax", include_self=True)
        ex = torch.exp(e - mx.gather(1, idx))
        den = torch.zeros(B,Nd,s.h, device=e.device).index_add_(1, dst, ex)
        alpha = ex / (den.gather(1, idx) + 1e-16)                                      # softmax over each node's neighbours
        msg = zs[:,src] * alpha.unsqueeze(-1)
        out = torch.zeros(B,Nd,s.h,s.f, device=e.device, dtype=msg.dtype).index_add_(1, dst, msg)
        return F.elu(out.reshape(B,Nd,-1))

TYPES = ["drug","gene","pathway"]
REL_NAMES = {"dg":("drug","gene"),"rdg":("gene","drug"),"gg":("gene","gene"),"gp":("gene","pathway"),
             "rgp":("pathway","gene"),"pp":("pathway","pathway"),"rpp":("pathway","pathway"),"dd":("drug","drug")}

class HANLayer(nn.Module):
    def __init__(s, hid, heads):
        super().__init__()
        s.rel = nn.ModuleDict({k: RelAttn(hid, heads) for k in REL_NAMES})
        s.sem = nn.ModuleDict({t: nn.Linear(hid, hid) for t in TYPES})
        s.q = nn.ParameterDict({t: nn.Parameter(torch.randn(hid) * 0.1) for t in TYPES})
        s.slf = nn.ModuleDict({t: nn.Linear(hid, hid) for t in TYPES})
    def forward(s, h, edges):                  # level 2: semantic attention across relations
        outs = {t: [] for t in TYPES}
        for name, ei in edges.items():
            if ei.size(1) == 0: continue
            a, b = REL_NAMES[name]; outs[b].append(s.rel[name](h[a], h[b], ei))
        new = {}
        for t in TYPES:
            zs = torch.stack(outs[t], 0)                                   # [R,B,N,hid]
            sc = (torch.tanh(s.sem[t](zs)) @ s.q[t]).mean(2)               # [R,B]
            beta = torch.softmax(sc, 0)[..., None, None]
            new[t] = F.elu(s.slf[t](h[t]) + (beta * zs).sum(0))
        return new

class Net(nn.Module):
    def __init__(s, G, NP, fps, hid=64, heads=4, use_graph=True):
        super().__init__(); s.use_graph = use_graph; s.register_buffer("fps", fps)
        s.fp_mlp = nn.Sequential(nn.Linear(1024,256), nn.ReLU(), nn.Linear(256,hid))
        s.expr_mlp = nn.Sequential(nn.Linear(G,256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256,hid))
        if use_graph:
            s.in_drug = nn.Linear(1024,hid); s.in_gene = nn.Linear(3,hid)
            s.path_emb = nn.Parameter(torch.randn(NP,hid) * 0.1)
            s.l1, s.l2 = HANLayer(hid,heads), HANLayer(hid,heads)
        s.head = nn.Sequential(nn.Linear(hid*(4 if use_graph else 2),128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128,1))
    def forward(s, cx, di, edges):             # cx [B,G,3], di [B]
        z = [s.fp_mlp(s.fps[di]), s.expr_mlp(cx[:,:,0])]
        if s.use_graph:
            B = cx.size(0)
            h = {"drug": s.in_drug(s.fps).unsqueeze(0).expand(B,-1,-1), "gene": s.in_gene(cx),
                 "pathway": s.path_emb.unsqueeze(0).expand(B,-1,-1)}
            h = s.l2(s.l1(h, edges), edges)
            z += [h["drug"][torch.arange(B, device=cx.device), di], h["gene"].mean(1)]
        return s.head(torch.cat(z, 1)).squeeze(-1)

def expand(b):
    """base edges -> dict of tensors with reverse edges. Same keys for real and shuffled graphs."""
    t = lambda a: torch.as_tensor(np.ascontiguousarray(a), dtype=torch.long, device=dev)
    rev = lambda a: a[::-1]
    return {"dg": t(b["dg"]), "rdg": t(rev(b["dg"])), "gg": t(np.concatenate([b["gg"], rev(b["gg"])], 1)),
            "gp": t(b["gp"]), "rgp": t(rev(b["gp"])), "pp": t(b["pp"]), "rpp": t(rev(b["pp"])),
            "dd": t(np.concatenate([b["dd"], rev(b["dd"])], 1))}

# Cell 4: training and evaluation

In [ ]:
CK, PR = "/kaggle/working/ckpt", "/kaggle/working/preds"
os.makedirs(CK, exist_ok=True); os.makedirs(PR, exist_ok=True)
EPOCHS, BS, MAXB, PATIENCE, NVAL = 8, 32, 300, 3, 3000

@torch.no_grad()
def predict(model, idx, edges, bs=64):
    model.eval(); out = []
    for i in range(0, len(idx), bs):
        b = torch.as_tensor(idx[i:i+bs], device=dev)
        out.append(model(CELLX[PC[b]], PD[b], edges).float().cpu())
    return torch.cat(out).numpy()

def metrics(idx, pred):
    d = df.iloc[idx].assign(pred=pred); pdr = {}
    for drug, g in d.groupby("drug_id"):
        if len(g) >= 30 and g.pred.std() > 0: pdr[int(drug)] = float(np.corrcoef(g.pred, g.label)[0,1])
    return {"pooled_r": float(np.corrcoef(pred, d.label)[0,1]), "rmse": float(np.sqrt(np.mean((pred - d.label.values)**2))),
            "per_drug_r": float(np.mean(list(pdr.values()))), "per_drug": pdr}

def train_eval(use_graph, edges, fold, seed, key, epochs=EPOCHS, bs=BS, maxb=MAXB, verbose=True):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    role = df[f"role_{fold}"].values
    tr, va, te = [np.where(role == k)[0] for k in ("train","val","test")]
    va_s = np.random.default_rng(123).choice(va, min(NVAL, len(va)), replace=False)   # fixed val subset (speed)
    mu, sd = Y[tr].mean(), Y[tr].std()                        # train-only statistics
    Yn = torch.tensor((Y - mu) / sd, device=dev)
    model = Net(G, NP, FPS, use_graph=use_graph).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    last_p, best_p = f"{CK}/{key}_last.pt", f"{CK}/{key}_best.pt"
    start, best, bad, hist = 0, 1e9, 0, []
    if os.path.exists(last_p):                                # resume an interrupted run
        ck = torch.load(last_p, map_location=dev)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        start, best, bad, hist = ck["epoch"], ck["best"], ck["bad"], ck["hist"]
        rng = np.random.default_rng(seed + start); print(f"  resumed {key} from epoch {start}")
    for ep in range(start, epochs):
        if bad >= PATIENCE: break
        model.train(); t0 = time.time(); losses = []
        perm = rng.permutation(tr)[:bs*maxb]
        for i in range(0, len(perm), bs):
            b = torch.as_tensor(perm[i:i+bs], device=dev)
            loss = F.mse_loss(model(CELLX[PC[b]], PD[b], edges), Yn[b])
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
            losses.append(loss.item())
        vl = float(np.mean((predict(model, va_s, edges) - ((Y[va_s] - mu) / sd))**2))
        hist.append({"epoch": ep+1, "train_mse": float(np.mean(losses)), "val_mse": vl})
        if vl < best:
            best, bad = vl, 0; torch.save(model.state_dict(), best_p)       # BEST checkpoint (by validation loss)
        else: bad += 1
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": ep+1,
                    "best": best, "bad": bad, "hist": hist}, last_p)         # resume checkpoint
        if verbose: print(f"  ep{ep+1} train={hist[-1]['train_mse']:.3f} val={vl:.4f} best={best:.4f} | {time.time()-t0:.0f}s")
    model.load_state_dict(torch.load(best_p, map_location=dev))
    pred = predict(model, te, edges) * sd + mu                # test touched once, with the best-VALIDATION weights
    np.save(f"{PR}/{key.replace('|','_')}.npy", pred)
    m = metrics(te, pred); m["history"] = hist; m["best_val_mse"] = best
    return m

# NOVELTY ( CELL 5 - SHUFFLING)

In [ ]:
import networkx as nx

def shuffle_undirected(ei, seed, factor=10):                   # PPI, degree-preserving (Maslov-Sneppen)
    g = nx.Graph(); g.add_edges_from(ei.T.tolist())
    nx.double_edge_swap(g, nswap=factor*g.number_of_edges(), max_tries=10**9, seed=seed)
    return np.sort(np.array(list(g.edges())).T, axis=0)

def shuffle_directed(ei, seed, homogeneous, factor=10):        # (u,v),(x,y) -> (u,y),(x,v)
    rng = np.random.default_rng(seed); src, dst = ei[0].copy(), ei[1].copy(); E = len(src)
    present = set(zip(src.tolist(), dst.tolist())); done = tries = 0
    while done < factor*E and tries < 200*factor*E:
        tries += 1; i, j = rng.integers(0, E, 2)
        u, v, x, y = src[i], dst[i], src[j], dst[j]
        if i == j or u == x or v == y: continue
        if homogeneous and (u == y or x == v): continue        # no self-loops
        if (u, y) in present or (x, v) in present: continue    # no duplicate edges
        present -= {(u, v), (x, y)}; present |= {(u, y), (x, v)}
        dst[i], dst[j] = y, v; done += 1
    assert done >= factor*E*0.9, f"only {done}/{factor*E} swaps achieved"
    return np.stack([src, dst])

def shuffle_rel(name, ei, seed):
    return shuffle_undirected(ei, seed) if name == "gg" else shuffle_directed(ei, seed, homogeneous=(name == "pp"))

def check_shuffle(name, old, new):
    ns, nd = SIZES[name]
    if name == "gg":
        d_old = np.bincount(old.ravel(), minlength=G); d_new = np.bincount(new.ravel(), minlength=G)
        assert (d_old == d_new).all()
    else:
        assert (np.bincount(old[0], minlength=ns) == np.bincount(new[0], minlength=ns)).all()
        assert (np.bincount(old[1], minlength=nd) == np.bincount(new[1], minlength=nd)).all()
    assert old.shape == new.shape and not (new[0] == new[1]).any() if name in ("gg","pp") else True
    key = lambda a: set(map(tuple, a.T.tolist()))
    kept = len(key(old) & key(new)) / old.shape[1]
    print(f"  {name}: degrees identical OK | edges changed: {1-kept:.1%} | no self-loops OK")

SHUF = {"B_shuf_all": ["dg","gg","gp","pp"], "B_shuf_dg": ["dg"], "B_shuf_gg": ["gg"], "B_shuf_gp_pp": ["gp","pp"]}
_cache = {}
def get_edges(arm, seed):
    if arm in ("A", "B_real"): return _cache.setdefault("real", expand(base))
    if (arm, seed) not in _cache:
        b = dict(base)
        for r in SHUF[arm]: b[r] = shuffle_rel(r, base[r], seed)
        _cache[(arm, seed)] = expand(b)
    return _cache[(arm, seed)]

print("Shuffle proof (seed 0, all biological relations):")
for r in SHUF["B_shuf_all"]: check_shuffle(r, base[r], shuffle_rel(r, base[r], 0))

# cell 6 ( time test)

In [ ]:
t0 = time.time()
r = train_eval(True, get_edges("B_real", 0), fold=0, seed=0, epochs=1, maxb=50)
print(f"50 batches incl. validation+test: {time.time()-t0:.0f}s | pooled r={r['pooled_r']:.3f}")
print("GPU memory peak (GB):", torch.cuda.max_memory_allocated()/1e9 if dev.type == "cuda" else "cpu")

# exp run ( cell 7)

In [ ]:
RES = "/kaggle/working/results.json"
R = json.load(open(RES)) if os.path.exists(RES) else {}

def run_all(arms, folds, seeds):
    for arm in arms:
        for seed in seeds:
            for f in folds:
                key = f"{arm}|fold{f}|seed{seed}"
                if key in R: continue
                t0 = time.time()
                m = train_eval(arm != "A", get_edges(arm, seed), f, seed, key, verbose=True)
                R[key] = m; json.dump(R, open(RES, "w"))
                print(f"DONE {key}: pooled r={m['pooled_r']:.3f} | per-drug r={m['per_drug_r']:.3f} | rmse={m['rmse']:.2f} | {time.time()-t0:.0f}s\n")

run_all(["A", "B_real", "B_shuf_all"], folds=[0], seeds=[0])     # first check: fold 0 only (~35 min)
# then, one fold at a time:  run_all([...], folds=[1], seeds=[0])  ... folds 2, 3, 4

# cell 8 stats

In [ ]:
from scipy.stats import wilcoxon
R = json.load(open(RES)); has_t = df.drop_duplicates("drug_id").set_index("drug_id").has_target.to_dict()

def arm_rows(arm): return [v for k, v in R.items() if k.split("|")[0] == arm]
print(f"{'arm':14s} {'runs':>4s} {'pooled r':>16s} {'per-drug r':>16s} {'RMSE':>12s}")
for arm in ["A","B_real","B_shuf_all","B_shuf_dg","B_shuf_gg","B_shuf_gp_pp"]:
    rows = arm_rows(arm)
    if rows:
        f = lambda k: (np.mean([r[k] for r in rows]), np.std([r[k] for r in rows]))
        print(f"{arm:14s} {len(rows):4d} {f('pooled_r')[0]:.3f}±{f('pooled_r')[1]:.3f}   {f('per_drug_r')[0]:.3f}±{f('per_drug_r')[1]:.3f}   {f('rmse')[0]:.2f}±{f('rmse')[1]:.2f}")

def drug_r(arm):
    acc = {}
    for r in arm_rows(arm):
        for d, v in r["per_drug"].items(): acc.setdefault(int(d), []).append(v)
    return {d: np.mean(v) for d, v in acc.items()}

def paired(x, y, subset=None, n_boot=5000):
    a, b = drug_r(x), drug_r(y)
    ds = [d for d in a if d in b and (subset is None or has_t.get(d) == subset)]
    if len(ds) < 10: return print(f"{x} vs {y}: too few common drugs ({len(ds)})")
    diff = np.array([a[d] - b[d] for d in ds]); rng = np.random.default_rng(0)
    bs = [rng.choice(diff, len(diff)).mean() for _ in range(n_boot)]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    print(f"{x:11s} - {y:11s} | n drugs={len(ds):3d} | mean diff in per-drug r = {diff.mean():+.3f} "
          f"[95% CI {lo:+.3f}, {hi:+.3f}] | Wilcoxon p={wilcoxon(diff).pvalue:.3g}")

for subset, label in [(None, "ALL drugs"), (True, "drugs WITH targets (90)"), (False, "drugs WITHOUT targets (82)")]:
    print("\n", label)
    for x, y in [("B_real","A"), ("B_real","B_shuf_all"), ("B_shuf_all","A")]: paired(x, y, subset)

# graphs

In [ ]:
import matplotlib.pyplot as plt
FIG = "/kaggle/working/figs"; os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "font.size": 11})
COL = {"A":"#8FA9DB","B_real":"#5FBF8F","B_shuf_all":"#E8836B","B_shuf_dg":"#F2B38F","B_shuf_gg":"#F2C98F","B_shuf_gp_pp":"#D9A0C8"}
NAME = {"A":"Arm A\n(no graph)","B_real":"Arm B\n(real graph)","B_shuf_all":"Arm B\n(shuffled graph)",
        "B_shuf_dg":"shuffle\ndrug-gene","B_shuf_gg":"shuffle\nPPI","B_shuf_gp_pp":"shuffle\npathways"}
R = json.load(open(RES))
def rows(a): return [v for k, v in R.items() if k.split("|")[0] == a]

# FIG 1: data funnel  (what we started with -> what we trained on)
fig, ax = plt.subplots(figsize=(8, 3.8))
lab = ["GitHub training table", "cell lines with expr+mut+CNV", "after averaging duplicates"]
val = [151973, 95316, 92109]
b = ax.barh(lab[::-1], val[::-1], color=["#5FBF8F", "#7F77DD", "#8FA9DB"])
for r_, v, c in zip(b, val[::-1], ["595 cell lines", "595 cell lines", "962 cell lines"][::-1]):
    ax.text(v + 1500, r_.get_y() + r_.get_height()/2, f"{v:,} pairs | {c}", va="center")
ax.set_xlim(0, 215000); ax.set_xlabel("drug x cell-line pairs"); ax.set_title("Data funnel (172 drugs throughout)")
plt.tight_layout(); plt.savefig(f"{FIG}/fig1_funnel.png"); plt.show()

# FIG 2: main result, per-drug correlation on UNSEEN drugs
arms = [a for a in ["A","B_real","B_shuf_all"] if rows(a)]
if arms:
    m = [np.mean([r["per_drug_r"] for r in rows(a)]) for a in arms]; s = [np.std([r["per_drug_r"] for r in rows(a)]) for a in arms]
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.bar([NAME[a] for a in arms], m, yerr=s, color=[COL[a] for a in arms], capsize=5)
    ax.axhline(0.626, ls="--", color="gray"); ax.text(len(arms)-0.5, 0.628, "cell-line-mean baseline (ignores the drug)", ha="right", va="bottom", fontsize=9)
    for i, v in enumerate(m): ax.text(i, v + 0.004, f"{v:.3f}", ha="center", fontweight="bold")
    ax.set_ylim(0.5, max(m) + 0.08); ax.set_ylabel("per-drug Pearson r (mean over test drugs)")
    ax.set_title("Prediction for drugs never seen in training"); plt.tight_layout(); plt.savefig(f"{FIG}/fig2_main_result.png"); plt.show()

# FIG 3: training curves (shows checkpointing: we keep the validation minimum)
fig, axs = plt.subplots(1, len(arms) or 1, figsize=(4.2*max(len(arms),1), 3.8), sharey=True); axs = np.atleast_1d(axs)
for ax, a in zip(axs, arms):
    k = f"{a}|fold0|seed0"
    if k in R:
        h = R[k]["history"]; ep = [x["epoch"] for x in h]; v = [x["val_mse"] for x in h]
        ax.plot(ep, [x["train_mse"] for x in h], label="train loss", color="#7F77DD"); ax.plot(ep, v, label="validation loss", color="#E8836B")
        bi = int(np.argmin(v)); ax.scatter(ep[bi], v[bi], s=90, color="gold", edgecolor="k", zorder=5, label="saved checkpoint")
    ax.set_title(NAME[a].replace("\n", " ")); ax.set_xlabel("epoch")
axs[0].set_ylabel("MSE (standardised LN_IC50)"); axs[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIG}/fig3_training_curves.png"); plt.show()

# FIG 4: every test drug, real graph vs shuffled graph
if rows("B_real") and rows("B_shuf_all"):
    a, b2 = drug_r("B_real"), drug_r("B_shuf_all"); ds = [d for d in a if d in b2]
    x = np.array([b2[d] for d in ds]); y = np.array([a[d] for d in ds]); t = np.array([has_t.get(d, False) for d in ds])
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.scatter(x[~t], y[~t], c="#8FA9DB", s=25, label="drug has no target in graph"); ax.scatter(x[t], y[t], c="#5FBF8F", s=30, label="drug has graph targets")
    lim = [min(x.min(), y.min()) - .03, max(x.max(), y.max()) + .03]; ax.plot(lim, lim, "k--", lw=1)
    ax.set_xlabel("per-drug r, shuffled graph"); ax.set_ylabel("per-drug r, real graph"); ax.legend(fontsize=8)
    ax.set_title("Above the line = real graph better for that drug"); plt.tight_layout(); plt.savefig(f"{FIG}/fig4_real_vs_shuffled_per_drug.png"); plt.show()

# FIG 5: paired difference with 95% bootstrap CI (the statistical answer)
def ci(x, y, subset):
    a, b2 = drug_r(x), drug_r(y); ds = [d for d in a if d in b2 and (subset is None or has_t.get(d) == subset)]
    if len(ds) < 10: return None
    diff = np.array([a[d] - b2[d] for d in ds]); rg = np.random.default_rng(0)
    bs = [rg.choice(diff, len(diff)).mean() for _ in range(3000)]
    return diff.mean(), *np.percentile(bs, [2.5, 97.5]), len(ds)
pairs = [("B_real","B_shuf_all","real - shuffled"), ("B_real","A","real - Arm A")]
subs = [(None,"all drugs"), (True,"drugs WITH targets"), (False,"drugs WITHOUT targets")]
fig, ax = plt.subplots(figsize=(7.5, 4.2)); w = 0.35; drawn = False
for j, (x, y, lbl) in enumerate(pairs):
    for i, (sub, sl) in enumerate(subs):
        r_ = ci(x, y, sub) if rows(x) and rows(y) else None
        if r_:
            ax.errorbar(i + j*w, r_[0], yerr=[[r_[0]-r_[1]], [r_[2]-r_[0]]], fmt="o", capsize=5, color=["#5FBF8F","#7F77DD"][j], label=lbl if i == 0 else None); drawn = True
ax.axhline(0, color="k", lw=1); ax.set_xticks([i + w/2 for i in range(3)]); ax.set_xticklabels([s[1] for s in subs])
ax.set_ylabel("difference in per-drug r (95% CI)"); ax.set_title("CI that excludes 0 = a real effect"); ax.legend()
plt.tight_layout(); plt.savefig(f"{FIG}/fig5_paired_difference.png"); plt.show()

# FIG 6: which edge type carries the signal (after the ablation runs)
ab = [a for a in ["A","B_real","B_shuf_all","B_shuf_dg","B_shuf_gg","B_shuf_gp_pp"] if rows(a)]
if len(ab) > 3:
    m = [np.mean([r["per_drug_r"] for r in rows(a)]) for a in ab]
    fig, ax = plt.subplots(figsize=(8.5, 4.3)); ax.bar([NAME[a] for a in ab], m, color=[COL[a] for a in ab])
    ax.set_ylim(min(m) - .02, max(m) + .02); ax.set_ylabel("per-drug r"); ax.set_title("Which part of the graph matters?")
    plt.tight_layout(); plt.savefig(f"{FIG}/fig6_edge_ablation.png"); plt.show()

# FIG 7: proof that the shuffle keeps degrees (works without training)
old, new = base["gg"], shuffle_rel("gg", base["gg"], 0)
d0 = np.bincount(old.ravel(), minlength=G); d1 = np.bincount(new.ravel(), minlength=G)
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].scatter(d0, d1, s=8, alpha=.4, color="#7F77DD"); axs[0].plot([0, d0.max()], [0, d0.max()], "k--", lw=1)
axs[0].set_xlabel("gene degree, real PPI"); axs[0].set_ylabel("gene degree, shuffled PPI"); axs[0].set_title("Every gene keeps its degree")
chg = [1 - len(set(map(tuple, base[r].T.tolist())) & set(map(tuple, shuffle_rel(r, base[r], 0).T.tolist()))) / base[r].shape[1] for r in ["dg","gg","gp","pp"]]
axs[1].bar(["drug-gene","PPI","gene-pathway","pathway\nhierarchy"], chg, color="#E8836B"); axs[1].set_ylim(0, 1); axs[1].set_ylabel("fraction of edges rewired"); axs[1].set_title("...but the wiring changes")
plt.tight_layout(); plt.savefig(f"{FIG}/fig7_shuffle_proof.png"); plt.show()

# FIG 8: predicted vs actual on unseen drugs, fold 0
pf = f"{PR}/B_real_fold0_seed0.npy"
if os.path.exists(pf):
    te = np.where(df.role_0.values == "test")[0]; p = np.load(pf); t_ = df.label.values[te]
    fig, ax = plt.subplots(figsize=(5.5, 5)); hb = ax.hexbin(t_, p, gridsize=45, cmap="viridis", mincnt=1, bins="log")
    lim = [t_.min(), t_.max()]; ax.plot(lim, lim, "w--"); fig.colorbar(hb, label="log count")
    ax.set_xlabel("actual LN_IC50"); ax.set_ylabel("predicted LN_IC50"); ax.set_title(f"Arm B, fold 0 (pooled r = {np.corrcoef(p, t_)[0,1]:.2f})")
    plt.tight_layout(); plt.savefig(f"{FIG}/fig8_pred_vs_actual.png"); plt.show()

In [ ]:
import os, glob
print("working:", os.listdir("/kaggle/working") if os.path.exists("/kaggle/working") else "missing")
print("inputs:", [p for p in glob.glob("/kaggle/input/**/*", recursive=True) if os.path.isfile(p)][:20])
try: print("variables alive:", len(df), G)
except Exception as e: print("session was reset:", type(e).__name__)

# **day 2**

In [ ]:
import glob, numpy as np
f = glob.glob("/kaggle/input/**/cell_features_gene_universe.npz", recursive=True)[0]
z = np.load(f); c = z["cnv"]
print(f, "| cells", len(z["cell_ids"]), "| genes", len(z["gene_ids"]))
print(f"deleted {np.mean(c==-1):.1%} | amplified {np.mean(c==1):.1%}")

In [ ]:
!pip install -q rdkit
import os, glob, subprocess, json, time, re, copy
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

REPO = "/kaggle/working/bioprior"
if not os.path.exists(REPO):
    r = subprocess.run(["git","clone","https://github.com/raveena2309/bioprior.git",REPO],
                       capture_output=True, text=True); print(r.stdout, r.stderr)
P = REPO + "/data/processed/"

def find(name):
    """Search all attached Kaggle datasets and /kaggle/working for a file."""
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"/kaggle/working/{name}")
    assert hits, f"{name} not found. Attach the dataset via Add Input."
    return hits[0]

FEAT = find("cell_features_gene_universe.npz")
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("repo:", os.path.exists(P + "training_table.csv"), "| features:", FEAT, "| device:", dev)


In [ ]:
tt = pd.read_csv(P + "training_table.csv")
dg = pd.read_csv(P + "graph_drug_gene_edges.csv")
cells = set(np.load(FEAT)["cell_ids"].tolist())

print("1) original:", len(tt), "rows |", tt.drug_id.nunique(), "drugs |", tt.cell_line_id.nunique(), "cell lines")
t1 = tt[tt.cell_line_id.isin(cells)].copy()
print("2) with full features:", len(t1), "rows |", t1.cell_line_id.nunique(), "cell lines")

df = (t1.groupby(["drug_id", "cell_line_id"], as_index=False)
        .agg(label=("label", "mean"), DRUG_NAME=("DRUG_NAME", "first"), chembl_id=("chembl_id", "first"),
             smiles=("smiles", "first"), PUTATIVE_TARGET=("PUTATIVE_TARGET", "first"),
             PATHWAY_NAME=("PATHWAY_NAME", "first")))
assert not df.duplicated(["drug_id", "cell_line_id"]).any()
assert np.isfinite(df["label"]).all() and df["smiles"].notna().all()

n_t = dg.groupby("chembl_id").gene_symbol.nunique()
df["n_targets"] = df.chembl_id.map(n_t).fillna(0).astype(int)
df["has_target"] = df.n_targets > 0
print("3) FINAL:", len(df), "pairs |", df.drug_id.nunique(), "drugs |", df.cell_line_id.nunique(), "cell lines")
print("   drugs with target:", df[df.has_target].drug_id.nunique())
df.to_csv("/kaggle/working/clean_table.csv", index=False)

In [ ]:
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import AllChem
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
RDLogger.DisableLog("rdApp.*")

df = pd.read_csv("/kaggle/working/clean_table.csv")
tt = pd.read_csv(P + "training_table.csv")
cells = set(np.load(FEAT)["cell_ids"].tolist())

t1 = tt[tt.cell_line_id.isin(cells)]
g = t1.groupby(["drug_id","cell_line_id"])["label"]
sp = (g.max() - g.min()).rename("spread").reset_index()
n0 = len(df); df = df.merge(sp, on=["drug_id","cell_line_id"], how="left"); assert len(df) == n0
df["spread"] = df.spread.fillna(0)

drugs = df.drop_duplicates("drug_id")[["drug_id","DRUG_NAME","smiles"]].reset_index(drop=True)
mols = [Chem.MolFromSmiles(s) for s in drugs.smiles]
assert all(m is not None for m in mols), "unparseable SMILES"
fps = [AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=1024) for m in mols]

THR, rr, cc = 0.5, [], []
for i in range(len(fps)-1):
    for k, s in enumerate(DataStructs.BulkTanimotoSimilarity(fps[i], fps[i+1:])):
        if s >= THR: rr.append(i); cc.append(i+1+k)
n = len(drugs)
n_fam, fam = connected_components(coo_matrix((np.ones(len(rr)), (rr, cc)), shape=(n, n)), directed=False)
drugs["family"] = fam
print(f"drugs {n} -> families {n_fam} | similar pairs >= {THR}: {len(rr)} | largest family: {np.bincount(fam).max()}")
df = df.merge(drugs[["drug_id","family"]], on="drug_id", how="left").reset_index(drop=True)
assert df.family.notna().all()

df["fold"] = -1
for f, (_, te) in enumerate(GroupKFold(n_splits=5).split(df, groups=df.family)):
    df.loc[te, "fold"] = f
assert (df.fold >= 0).all()

for f in range(5):
    role = np.where(df.fold == f, "test", "train").astype(object)
    tr = np.where(df.fold != f)[0]
    _, va = next(GroupShuffleSplit(1, test_size=0.1, random_state=f).split(tr, groups=df.family.values[tr]))
    role[tr[va]] = "val"
    df[f"role_{f}"] = role

assert not df.duplicated(["drug_id","cell_line_id"]).any()
assert df.groupby("drug_id").fold.nunique().max() == 1
for f in range(5):
    r = df[f"role_{f}"]
    D = {k: set(df.loc[r==k,"drug_id"]) for k in ["train","val","test"]}
    Fm = {k: set(df.loc[r==k,"family"]) for k in ["train","val","test"]}
    assert not (D["train"]&D["test"]) and not (D["val"]&D["test"]) and not (D["train"]&D["val"])
    assert not (Fm["train"]&Fm["test"]) and not (Fm["val"]&Fm["test"]) and not (Fm["train"]&Fm["val"])
    te = df[r=="test"]
    print(f"fold {f}: drugs {len(D['train'])}/{len(D['val'])}/{len(D['test'])} | pairs {(r=='train').sum()}/{(r=='val').sum()}/{(r=='test').sum()} | test drugs with target: {te[te.has_target].drug_id.nunique()}")
print("ALL LEAKAGE CHECKS PASSED")
df.to_csv("/kaggle/working/clean_table_folds.csv", index=False)
print("saved clean_table_folds.csv  <-- DOWNLOAD IT")

In [ ]:
SEL_G = 1000
df = pd.read_csv("/kaggle/working/clean_table_folds.csv")
feat = np.load(FEAT)
cell_ids = feat["cell_ids"].tolist(); genes_all = feat["gene_ids"].astype(int)
assert (np.diff(genes_all) > 0).all()

hdr = pd.read_csv(find("OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv"), nrows=0).columns
sym2ent = {m.group(1): int(m.group(2)) for c in hdr if (m := re.match(r"^(.+?) \((\d+)\)$", str(c)))}

drug_ids = sorted(df.drug_id.unique()); didx = {d:i for i,d in enumerate(drug_ids)}; D = len(drug_ids)

dgc = pd.read_csv(P + "graph_drug_gene_edges.csv")
dgc["gene_id"] = dgc.gene_symbol.map(sym2ent)
print(f"ChEMBL target symbols: {dgc.gene_symbol.nunique()} | mapped: {dgc.dropna(subset=['gene_id']).gene_symbol.nunique()}")
dgc = dgc.dropna(subset=["gene_id"]); dgc["gene_id"] = dgc.gene_id.astype(int)
dmeta = df.drop_duplicates("drug_id")[["drug_id","chembl_id"]]
dge = dmeta.merge(dgc[["chembl_id","gene_id"]], on="chembl_id")[["drug_id","gene_id"]].drop_duplicates()
univ = set(genes_all)
dge_u = dge[dge.gene_id.isin(univ)]
print(f"drug-gene edges: {len(dge)} -> inside universe: {len(dge_u)} | drugs with targets: {dge_u.drug_id.nunique()}/{D}")

ppi = pd.read_csv(P + "graph_gene_gene_edges.csv"); n_ppi_raw = len(ppi)
ppi = ppi[ppi.gene_a.isin(univ) & ppi.gene_b.isin(univ) & (ppi.gene_a != ppi.gene_b)]
und = np.unique(np.sort(ppi[["gene_a","gene_b"]].values, axis=1), axis=0)
deg = pd.concat([pd.Series(und[:,0]), pd.Series(und[:,1])]).value_counts()
tgt = sorted(set(dge_u.gene_id)); tgt_set = set(tgt)
fill = [g for g in deg.index if g not in tgt_set][:max(0, SEL_G - len(tgt))]
genes = sorted(tgt_set | set(fill)); G = len(genes); gidx = {g:i for i,g in enumerate(genes)}
print(f"genes: {len(univ)} -> G={G} (targets {len(tgt)} + top-degree {len(fill)})")

gg  = np.array([[gidx[a], gidx[b]] for a, b in und if a in gidx and b in gidx]).T
dgE = np.array([[didx[d], gidx[g]] for d, g in dge_u[dge_u.gene_id.isin(gidx)].values]).T

gp_df = pd.read_csv(P + "graph_gene_pathway_edges.csv"); gp_df = gp_df[gp_df.gene_id.isin(gidx)]
cnt = gp_df.groupby("reactome_id").gene_id.nunique()
keep = cnt[cnt >= 5].index; pidx = {p:i for i,p in enumerate(keep)}; NP = len(pidx)
gp_df = gp_df[gp_df.reactome_id.isin(pidx)]
gpE = np.array([[gidx[g], pidx[p]] for g, p in gp_df[["gene_id","reactome_id"]].values]).T
ph = pd.read_csv(P + "graph_pathway_hierarchy_edges.csv")
ph = ph[ph.parent_pathway.isin(pidx) & ph.child_pathway.isin(pidx)]
ppE = np.array([[pidx[a], pidx[b]] for a, b in ph.values]).T

dd = pd.read_csv(P + "graph_drug_drug_edges.csv"); dd = dd[dd.drug_a.isin(didx) & dd.drug_b.isin(didx)]
ddE = np.array([[didx[a], didx[b]] for a, b in dd[["drug_a","drug_b"]].values]).T

base = {"dg": dgE, "gg": gg, "gp": gpE, "pp": ppE, "dd": ddE}
SIZES = {"dg":(D,G), "gg":(G,G), "gp":(G,NP), "pp":(NP,NP), "dd":(D,D)}
print(f"PPI {n_ppi_raw:,}->{gg.shape[1]:,} | gene-pathway 49,642->{gpE.shape[1]:,} ({NP} pathways) | hierarchy 2,899->{ppE.shape[1]} | drug-gene 347->{dgE.shape[1]} | drug-drug 5->{ddE.shape[1]}")
assert all(e.shape[0] == 2 and e.shape[1] > 0 for e in base.values())

pos = np.searchsorted(genes_all, genes)
CELLX = torch.tensor(np.stack([feat["expr"][:,pos], feat["mut"][:,pos], feat["cnv"][:,pos]], -1).astype("float32"), device=dev)
fp_np = np.stack([np.array(AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(
        df[df.drug_id == d].smiles.iloc[0]), 2, nBits=1024), dtype="float32") for d in drug_ids])
FPS = torch.tensor(fp_np, device=dev)
cmap = {c:i for i,c in enumerate(cell_ids)}; assert df.cell_line_id.isin(cmap).all()
PC = torch.tensor(df.cell_line_id.map(cmap).values, device=dev)
PD = torch.tensor(df.drug_id.map(didx).values, device=dev)
Y = df.label.values.astype("float32")
print("CELLX", tuple(CELLX.shape), "| FPS", tuple(FPS.shape), "| pairs", len(df))

In [ ]:
class RelAttn(nn.Module):                      # level 1: node attention inside ONE relation
    def __init__(s, hid, heads):
        super().__init__(); s.h, s.f = heads, hid // heads
        s.Ws = nn.Linear(hid, hid, bias=False); s.Wd = nn.Linear(hid, hid, bias=False)
        s.a_s = nn.Parameter(torch.randn(1,1,heads,s.f) * 0.1); s.a_d = nn.Parameter(torch.randn(1,1,heads,s.f) * 0.1)
    def forward(s, hs, hd, ei):
        B, Nd = hs.size(0), hd.size(1); src, dst = ei
        zs = s.Ws(hs).view(B,-1,s.h,s.f); zd = s.Wd(hd).view(B,-1,s.h,s.f)
        e = F.leaky_relu((zs*s.a_s).sum(-1)[:,src] + (zd*s.a_d).sum(-1)[:,dst], 0.2)
        idx = dst.view(1,-1,1).expand_as(e)
        mx = torch.full((B,Nd,s.h), -1e9, device=e.device).scatter_reduce(1, idx, e.detach(), reduce="amax", include_self=True)
        ex = torch.exp(e - mx.gather(1, idx))
        den = torch.zeros(B,Nd,s.h, device=e.device).index_add_(1, dst, ex)
        alpha = ex / (den.gather(1, idx) + 1e-16)
        msg = zs[:,src] * alpha.unsqueeze(-1)
        out = torch.zeros(B,Nd,s.h,s.f, device=e.device, dtype=msg.dtype).index_add_(1, dst, msg)
        return F.elu(out.reshape(B,Nd,-1))

TYPES = ["drug","gene","pathway"]
REL_NAMES = {"dg":("drug","gene"),"rdg":("gene","drug"),"gg":("gene","gene"),"gp":("gene","pathway"),
             "rgp":("pathway","gene"),"pp":("pathway","pathway"),"rpp":("pathway","pathway"),"dd":("drug","drug")}

class HANLayer(nn.Module):                     # level 2: semantic attention across relations
    def __init__(s, hid, heads):
        super().__init__()
        s.rel = nn.ModuleDict({k: RelAttn(hid, heads) for k in REL_NAMES})
        s.sem = nn.ModuleDict({t: nn.Linear(hid, hid) for t in TYPES})
        s.q = nn.ParameterDict({t: nn.Parameter(torch.randn(hid) * 0.1) for t in TYPES})
        s.slf = nn.ModuleDict({t: nn.Linear(hid, hid) for t in TYPES})
    def forward(s, h, edges):
        outs = {t: [] for t in TYPES}
        for name, ei in edges.items():
            if ei.size(1) == 0: continue
            a, b = REL_NAMES[name]; outs[b].append(s.rel[name](h[a], h[b], ei))
        new = {}
        for t in TYPES:
            zs = torch.stack(outs[t], 0)
            sc = (torch.tanh(s.sem[t](zs)) @ s.q[t]).mean(2)
            beta = torch.softmax(sc, 0)[..., None, None]
            new[t] = F.elu(s.slf[t](h[t]) + (beta * zs).sum(0))
        return new

class Net(nn.Module):
    def __init__(s, G, NP, fps, hid=64, heads=4, use_graph=True):
        super().__init__(); s.use_graph = use_graph; s.register_buffer("fps", fps)
        s.fp_mlp = nn.Sequential(nn.Linear(1024,256), nn.ReLU(), nn.Linear(256,hid))
        s.expr_mlp = nn.Sequential(nn.Linear(G,256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256,hid))
        if use_graph:
            s.in_drug = nn.Linear(1024,hid); s.in_gene = nn.Linear(3,hid)
            s.path_emb = nn.Parameter(torch.randn(NP,hid) * 0.1)
            s.l1, s.l2 = HANLayer(hid,heads), HANLayer(hid,heads)
        s.head = nn.Sequential(nn.Linear(hid*(4 if use_graph else 2),128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128,1))
    def forward(s, cx, di, edges):
        z = [s.fp_mlp(s.fps[di]), s.expr_mlp(cx[:,:,0])]
        if s.use_graph:
            B = cx.size(0)
            h = {"drug": s.in_drug(s.fps).unsqueeze(0).expand(B,-1,-1), "gene": s.in_gene(cx),
                 "pathway": s.path_emb.unsqueeze(0).expand(B,-1,-1)}
            h = s.l2(s.l1(h, edges), edges)
            z += [h["drug"][torch.arange(B, device=cx.device), di], h["gene"].mean(1)]
        return s.head(torch.cat(z, 1)).squeeze(-1)

def expand(b):
    t = lambda a: torch.as_tensor(np.ascontiguousarray(a), dtype=torch.long, device=dev)
    rev = lambda a: a[::-1]
    return {"dg": t(b["dg"]), "rdg": t(rev(b["dg"])), "gg": t(np.concatenate([b["gg"], rev(b["gg"])], 1)),
            "gp": t(b["gp"]), "rgp": t(rev(b["gp"])), "pp": t(b["pp"]), "rpp": t(rev(b["pp"])),
            "dd": t(np.concatenate([b["dd"], rev(b["dd"])], 1))}

In [ ]:
import networkx as nx

def shuffle_undirected(ei, seed, factor=10):
    g = nx.Graph(); g.add_edges_from(ei.T.tolist())
    nx.double_edge_swap(g, nswap=factor*g.number_of_edges(), max_tries=10**9, seed=seed)
    return np.sort(np.array(list(g.edges())).T, axis=0)

def shuffle_directed(ei, seed, homogeneous, factor=10):
    rng = np.random.default_rng(seed); src, dst = ei[0].copy(), ei[1].copy(); E = len(src)
    present = set(zip(src.tolist(), dst.tolist())); done = tries = 0
    while done < factor*E and tries < 200*factor*E:
        tries += 1; i, j = rng.integers(0, E, 2)
        u, v, x, y = src[i], dst[i], src[j], dst[j]
        if i == j or u == x or v == y: continue
        if homogeneous and (u == y or x == v): continue
        if (u, y) in present or (x, v) in present: continue
        present -= {(u, v), (x, y)}; present |= {(u, y), (x, v)}
        dst[i], dst[j] = y, v; done += 1
    assert done >= factor*E*0.9, f"only {done}/{factor*E} swaps"
    return np.stack([src, dst])

def shuffle_rel(name, ei, seed):
    return shuffle_undirected(ei, seed) if name == "gg" else shuffle_directed(ei, seed, homogeneous=(name == "pp"))

def check_shuffle(name, old, new):
    ns, nd = SIZES[name]
    if name == "gg":
        assert (np.bincount(old.ravel(), minlength=G) == np.bincount(new.ravel(), minlength=G)).all()
    else:
        assert (np.bincount(old[0], minlength=ns) == np.bincount(new[0], minlength=ns)).all()
        assert (np.bincount(old[1], minlength=nd) == np.bincount(new[1], minlength=nd)).all()
    assert old.shape == new.shape
    if name in ("gg","pp"): assert not (new[0] == new[1]).any()
    key = lambda a: set(map(tuple, a.T.tolist()))
    print(f"  {name}: degrees identical OK | edges changed: {1 - len(key(old) & key(new))/old.shape[1]:.1%}")

SHUF = {"B_shuf_all": ["dg","gg","gp","pp"], "B_shuf_dg": ["dg"], "B_shuf_gg": ["gg"], "B_shuf_gp_pp": ["gp","pp"]}
_cache = {}
def get_edges(arm, seed):
    if arm in ("A", "B_real"): return _cache.setdefault("real", expand(base))
    if (arm, seed) not in _cache:
        b = dict(base)
        for r in SHUF[arm]: b[r] = shuffle_rel(r, base[r], seed)
        _cache[(arm, seed)] = expand(b)
    return _cache[(arm, seed)]

print("Shuffle proof:")
for r in SHUF["B_shuf_all"]: check_shuffle(r, base[r], shuffle_rel(r, base[r], 0))

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
FIG = "/kaggle/working/figs"; os.makedirs(FIG, exist_ok=True)
CK, PR = "/kaggle/working/ckpt_v2", "/kaggle/working/preds_v2"
os.makedirs(CK, exist_ok=True); os.makedirs(PR, exist_ok=True)
RES = "/kaggle/working/results_v2.json"
EPOCHS, BS, MAXB, PATIENCE, NVAL = 8, 32, 300, 3, 6000

def load_R(path): return json.load(open(path)) if os.path.exists(path) else {}
def rows(R, arm): return [v for k, v in R.items() if k.split("|")[0] == arm]
def drug_r(R, arm):
    acc = {}
    for r in rows(R, arm):
        for d, v in r["per_drug"].items(): acc.setdefault(int(d), []).append(v)
    return {d: float(np.mean(v)) for d, v in acc.items()}
has_t = df.drop_duplicates("drug_id").set_index("drug_id").has_target.to_dict()
R2 = load_R(RES)

def per_drug(idx, pred, min_n=30):
    d = df.iloc[idx].assign(pred=np.asarray(pred)); out = {}
    for drug, g in d.groupby("drug_id"):
        if len(g) >= min_n and g.pred.std() > 0: out[int(drug)] = float(np.corrcoef(g.pred, g.label)[0, 1])
    return {"per_drug": out, "per_drug_r": float(np.mean(list(out.values())))}

def prep_fold(fold):
    role = df[f"role_{fold}"].values
    tr, va, te = [np.where(role == k)[0] for k in ("train", "val", "test")]
    dm = pd.Series(Y[tr]).groupby(df.drug_id.values[tr]).transform("mean").values
    yc = Y[tr] - dm
    c = df.cell_line_id.values[tr]
    S = pd.Series(yc).groupby(c).sum(); N = pd.Series(yc).groupby(c).count()
    b_full = S / N
    b_loo = (S.reindex(c).values - yc) / np.maximum(N.reindex(c).values - 1, 1)
    t = np.zeros(len(df), "float32"); t[tr] = yc - b_loo
    sd = float(t[tr].std())
    B = df.cell_line_id.map(b_full).fillna(0).values.astype("float32")
    return tr, va, te, t / sd, sd, B

@torch.no_grad()
def predict(model, idx, edges, bs=64):
    model.eval(); out = []
    for i in range(0, len(idx), bs):
        b = torch.as_tensor(idx[i:i+bs], device=dev)
        out.append(model(CELLX[PC[b]], PD[b], edges).float().cpu())
    return torch.cat(out).numpy()

def train_eval2(use_graph, edges, fold, seed, key, verbose=True):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    tr, va, te, T, sd, B = prep_fold(fold)
    va_s = np.random.default_rng(123).choice(va, min(NVAL, len(va)), replace=False)
    Tt = torch.tensor(T, device=dev)
    model = Net(G, NP, FPS, use_graph=use_graph).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    last_p, best_p = f"{CK}/{key}_last.pt", f"{CK}/{key}_best.pt"
    start, best, bad, hist = 0, -1e9, 0, []
    if os.path.exists(last_p):
        ck = torch.load(last_p, map_location=dev); model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        start, best, bad, hist = ck["epoch"], ck["best"], ck["bad"], ck["hist"]; rng = np.random.default_rng(seed + start)
        print(f"  resumed {key} from epoch {start}")
    base_val = per_drug(va_s, B[va_s])["per_drug_r"]
    for ep in range(start, EPOCHS):
        if bad >= PATIENCE: break
        model.train(); t0 = time.time(); losses = []
        perm = rng.permutation(tr)[:BS * MAXB]
        for i in range(0, len(perm), BS):
            b = torch.as_tensor(perm[i:i+BS], device=dev)
            loss = F.mse_loss(model(CELLX[PC[b]], PD[b], edges), Tt[b])
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
            losses.append(loss.item())
        vr = per_drug(va_s, B[va_s] + predict(model, va_s, edges) * sd)["per_drug_r"]
        hist.append({"epoch": ep + 1, "train_mse": float(np.mean(losses)), "val_r": vr, "val_base_r": base_val})
        if vr > best: best, bad = vr, 0; torch.save(model.state_dict(), best_p)
        else: bad += 1
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": ep + 1, "best": best, "bad": bad, "hist": hist}, last_p)
        if verbose: print(f"  ep{ep+1} train={hist[-1]['train_mse']:.3f} val_r={vr:.4f} (baseline {base_val:.4f}) best={best:.4f} | {time.time()-t0:.0f}s")
    model.load_state_dict(torch.load(best_p, map_location=dev))
    pred = B[te] + predict(model, te, edges) * sd
    np.save(f"{PR}/{key.replace('|','_')}.npy", pred)
    m = per_drug(te, pred); m["history"] = hist; m["best_val_r"] = best
    return m

def run_all2(arms, folds, seed=0):
    for f in folds:
        kb = f"BASE|fold{f}|seed{seed}"
        if kb not in R2:
            _, _, te, _, _, B = prep_fold(f); R2[kb] = per_drug(te, B[te]); json.dump(R2, open(RES, "w"))
            print(f"{kb}: per-drug r={R2[kb]['per_drug_r']:.3f}")
        for arm in arms:
            key = f"{arm}|fold{f}|seed{seed}"
            if key in R2: continue
            t0 = time.time()
            m = train_eval2(arm != "A", get_edges(arm, seed), f, seed, key)
            R2[key] = m; json.dump(R2, open(RES, "w"))
            print(f"DONE {key}: per-drug r={m['per_drug_r']:.3f} | {time.time()-t0:.0f}s\n")

In [ ]:
run_all2(["A", "B_real", "B_shuf_all"], folds=[0])        # development fold, about 30 min

In [ ]:
import os
print("results exists:", os.path.exists("/kaggle/working/results_v2.json"))
print("ckpts:", sorted(os.listdir("/kaggle/working/ckpt_v2"))[-8:] if os.path.exists("/kaggle/working/ckpt_v2") else "missing")
try:
    print("variables alive:", len(df), G, "| finished runs:", [k for k in R2 if "fold1" in k])
except Exception as e:
    print("session was reset:", type(e).__name__)

In [ ]:
for f in [1, 2, 3, 4]:
    run_all2(["A", "B_real", "B_shuf_all"], folds=[f])
    !cd /kaggle/working && zip -qr backup.zip results_v2.json clean_table_folds.csv preds_v2
    print(f"=== fold {f} finished and backed up ===\n")

In [ ]:
for f in range(5): run_all2(["B_shuf_dg", "B_shuf_gg", "B_shuf_gp_pp"], folds=[f])

In [ ]:
R2 = load_R(RES)
col = {"BASE":"#BBBBBB","A":"#8FA9DB","B_real":"#5FBF8F","B_shuf_all":"#E8836B","B_shuf_dg":"#F2B38F","B_shuf_gg":"#F2C98F","B_shuf_gp_pp":"#D9A0C8"}
nm = {"BASE":"cell-line mean\n(ignores drug)","A":"Arm A\nno graph","B_real":"Arm B\nreal graph","B_shuf_all":"Arm B\nshuffled graph",
      "B_shuf_dg":"shuffle\ndrug-gene","B_shuf_gg":"shuffle\nPPI","B_shuf_gp_pp":"shuffle\npathways"}

# ---------- statistics ----------
def paired(x, y, subset=None, n_boot=5000):
    a, b = drug_r(R2, x), drug_r(R2, y)
    ds = [d for d in a if d in b and (subset is None or has_t.get(d) == subset)]
    if len(ds) < 10: return print(f"{x} vs {y}: too few drugs ({len(ds)})")
    diff = np.array([a[d] - b[d] for d in ds]); g = np.random.default_rng(0)
    bs = [g.choice(diff, len(diff)).mean() for _ in range(n_boot)]; lo, hi = np.percentile(bs, [2.5, 97.5])
    print(f"{x:10s} - {y:10s} | n={len(ds):3d} | mean diff {diff.mean():+.4f} [95% CI {lo:+.4f}, {hi:+.4f}] | Wilcoxon p={wilcoxon(diff).pvalue:.3g}")
print(f"{'arm':13s} runs  per-drug r (mean ± SD over folds)")
for a in ["BASE","A","B_real","B_shuf_all","B_shuf_dg","B_shuf_gg","B_shuf_gp_pp"]:
    v = [r["per_drug_r"] for r in rows(R2, a)]
    if v: print(f"{a:13s} {len(v):3d}   {np.mean(v):.4f} ± {np.std(v):.4f}")
for sub, lab in [(None,"ALL drugs"),(True,"drugs WITH graph targets"),(False,"drugs WITHOUT targets (control)")]:
    print("\n", lab)
    for x, y in [("A","BASE"),("B_real","BASE"),("B_real","A"),("B_real","B_shuf_all")]:
        if rows(R2, x) and rows(R2, y): paired(x, y, sub)

# ---------- FIG 1: data funnel ----------
fig, ax = plt.subplots(figsize=(8.5, 3.6))
steps = [("GitHub training table",151973,962),("keep cell lines with expr+mut+CNV",95316,595),("average duplicate pairs",92109,595)]
for i,(lab,n_,c_) in enumerate(steps):
    ax.barh(i, n_, color=["#8FA9DB","#7F77DD","#5FBF8F"][i]); ax.text(n_+2000, i, f"{n_:,} pairs | {c_} cell lines", va="center")
ax.set_yticks(range(3)); ax.set_yticklabels([s[0] for s in steps]); ax.invert_yaxis()
ax.set_xlim(0,260000); ax.set_xticks([0,50000,100000,150000]); ax.set_xlabel("drug x cell-line pairs"); ax.set_title("Data funnel (172 drugs throughout)")
plt.tight_layout(); plt.savefig(f"{FIG}/fig1_funnel.png"); plt.show()

# ---------- FIG 2: main result ----------
arms = [a for a in ["BASE","A","B_real","B_shuf_all"] if rows(R2, a)]
if arms:
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    for i, a in enumerate(arms):
        v = [r["per_drug_r"] for r in rows(R2, a)]
        ax.bar(i, np.mean(v), color=col[a], alpha=.75); ax.scatter([i]*len(v), v, color="k", s=18, zorder=3)
        ax.text(i, np.mean(v)+.004, f"{np.mean(v):.3f}", ha="center", fontweight="bold")
    ax.set_xticks(range(len(arms))); ax.set_xticklabels([nm[a] for a in arms])
    lo = min(r["per_drug_r"] for a in arms for r in rows(R2, a)); ax.set_ylim(lo-.05, None)
    ax.set_ylabel("per-drug r on unseen drugs"); ax.set_title("Each dot = one test fold (about 34 unseen drugs)")
    plt.tight_layout(); plt.savefig(f"{FIG}/fig2_main_result.png"); plt.show()

# ---------- FIG 3: training curves with the baseline line ----------
ta = [a for a in ["A","B_real","B_shuf_all"] if f"{a}|fold0|seed0" in R2]
if ta:
    fig, axs = plt.subplots(1, len(ta), figsize=(4.4*len(ta), 3.8), sharey=True); axs = np.atleast_1d(axs)
    for ax, a in zip(axs, ta):
        h = R2[f"{a}|fold0|seed0"]["history"]; ep = [x["epoch"] for x in h]; v = [x["val_r"] for x in h]
        ax.plot(ep, v, color="#E8836B", label="validation per-drug r"); ax.axhline(h[0]["val_base_r"], ls="--", color="gray", label="cell-line-mean baseline")
        bi = int(np.argmax(v)); ax.scatter(ep[bi], v[bi], s=90, color="gold", edgecolor="k", zorder=5, label="saved checkpoint")
        ax.set_title(nm[a].replace("\n"," ")); ax.set_xlabel("epoch")
    axs[0].set_ylabel("validation per-drug r (fold 0)"); axs[0].legend(fontsize=8)
    plt.tight_layout(); plt.savefig(f"{FIG}/fig3_training_curves.png"); plt.show()

# ---------- FIG 4: per drug, real vs shuffled ----------
if rows(R2,"B_real") and rows(R2,"B_shuf_all"):
    a, b2 = drug_r(R2,"B_real"), drug_r(R2,"B_shuf_all"); ds = [d for d in a if d in b2]
    x = np.array([b2[d] for d in ds]); y = np.array([a[d] for d in ds]); t = np.array([has_t.get(d, False) for d in ds])
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.scatter(x[~t], y[~t], c="#8FA9DB", s=25, label="drug has no graph target"); ax.scatter(x[t], y[t], c="#5FBF8F", s=30, label="drug has graph targets")
    lim = [min(x.min(), y.min())-.03, max(x.max(), y.max())+.03]; ax.plot(lim, lim, "k--", lw=1)
    ax.set_xlabel("per-drug r, shuffled graph"); ax.set_ylabel("per-drug r, real graph"); ax.legend(fontsize=8)
    ax.set_title("Above the line = real graph better"); plt.tight_layout(); plt.savefig(f"{FIG}/fig4_real_vs_shuffled.png"); plt.show()

# ---------- FIG 5: paired difference with 95% CI ----------
def ci(x, y, subset):
    a, b2 = drug_r(R2,x), drug_r(R2,y); ds = [d for d in a if d in b2 and (subset is None or has_t.get(d) == subset)]
    if len(ds) < 10: return None
    diff = np.array([a[d]-b2[d] for d in ds]); g = np.random.default_rng(0)
    bs = [g.choice(diff, len(diff)).mean() for _ in range(3000)]
    return diff.mean(), *np.percentile(bs, [2.5, 97.5])
subs = [(None,"all drugs"),(True,"drugs WITH targets"),(False,"drugs WITHOUT targets")]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for j,(x,y,lbl) in enumerate([("B_real","B_shuf_all","real - shuffled"),("B_real","A","real - Arm A")]):
    for i,(sub,sl) in enumerate(subs):
        r_ = ci(x, y, sub) if rows(R2,x) and rows(R2,y) else None
        if r_: ax.errorbar(i+j*.3, r_[0], yerr=[[r_[0]-r_[1]],[r_[2]-r_[0]]], fmt="o", capsize=5, color=["#5FBF8F","#7F77DD"][j], label=lbl if i==0 else None)
ax.axhline(0, color="k", lw=1); ax.set_xticks([i+.15 for i in range(3)]); ax.set_xticklabels([s[1] for s in subs])
ax.set_ylabel("difference in per-drug r (95% CI)"); ax.set_title("CI that excludes 0 = a real effect"); ax.legend()
plt.tight_layout(); plt.savefig(f"{FIG}/fig5_paired_difference.png"); plt.show()

# ---------- FIG 6: edge-type ablation ----------
ab = [a for a in ["A","B_real","B_shuf_all","B_shuf_dg","B_shuf_gg","B_shuf_gp_pp"] if rows(R2, a)]
if len(ab) > 3:
    m = [np.mean([r["per_drug_r"] for r in rows(R2,a)]) for a in ab]
    fig, ax = plt.subplots(figsize=(8.5, 4.3)); ax.bar([nm[a] for a in ab], m, color=[col[a] for a in ab])
    ax.set_ylim(min(m)-.01, max(m)+.01); ax.set_ylabel("per-drug r"); ax.set_title("Which part of the graph matters?")
    plt.tight_layout(); plt.savefig(f"{FIG}/fig6_edge_ablation.png"); plt.show()

# ---------- FIG 7: shuffle proof ----------
old, new = base["gg"], shuffle_rel("gg", base["gg"], 0)
d0 = np.bincount(old.ravel(), minlength=G); d1 = np.bincount(new.ravel(), minlength=G)
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].scatter(d0, d1, s=8, alpha=.4, color="#7F77DD"); axs[0].plot([0, d0.max()], [0, d0.max()], "k--", lw=1)
axs[0].set_xlabel("gene degree, real PPI"); axs[0].set_ylabel("gene degree, shuffled PPI"); axs[0].set_title("Every gene keeps its degree")
key = lambda a: set(map(tuple, a.T.tolist()))
chg = [1 - len(key(base[r]) & key(shuffle_rel(r, base[r], 0)))/base[r].shape[1] for r in ["dg","gg","gp","pp"]]
axs[1].bar(["drug-gene","PPI","gene-pathway","pathway\nhierarchy"], chg, color="#E8836B"); axs[1].set_ylim(0,1)
axs[1].set_ylabel("fraction of edges rewired"); axs[1].set_title("...but the wiring changes")
plt.tight_layout(); plt.savefig(f"{FIG}/fig7_shuffle_proof.png"); plt.show()